In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import json
import joblib

In [57]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
# from sklearn.
from scipy.stats import loguniform

In [4]:
df = pd.read_parquet('../data/aircraft engine/PM_train.parquet')
df.head()

,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,s6,s7,s8,s9,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21,max,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,192,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,192,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,192,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,192,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.00,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,192,187


In [5]:
df['setting2'].value_counts()

setting2
-0.0003    2104
 0.0001    2097
 0.0000    2070
 0.0003    2065
-0.0004    2051
-0.0002    2049
 0.0002    2038
-0.0001    2029
 0.0004    1997
 0.0005    1068
-0.0005     958
 0.0006      71
-0.0006      34
Name: count, dtype: int64

In [6]:
df['s6'].value_counts()
# s6
# 21.61    20225
# 21.60      406
# Name: count, dtype: int64

s6
21.61    20225
21.60      406
Name: count, dtype: int64

In [7]:
df.drop(['id','setting3', 's1', 's5', 's6', 's10', 's16', 's18', 's19', 'cycle', 'max'], axis=1, inplace=True)

In [8]:
col = df.columns.tolist()

In [9]:
scaler = StandardScaler()
# df = scaler.fit_transform(df)
# df = pd.DataFrame(df, columns=col)
# df.head()

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   setting1  20631 non-null  float64
 1   setting2  20631 non-null  float64
 2   s2        20631 non-null  float64
 3   s3        20631 non-null  float64
 4   s4        20631 non-null  float64
 5   s7        20631 non-null  float64
 6   s8        20631 non-null  float64
 7   s9        20631 non-null  float64
 8   s11       20631 non-null  float64
 9   s12       20631 non-null  float64
 10  s13       20631 non-null  float64
 11  s14       20631 non-null  float64
 12  s15       20631 non-null  float64
 13  s17       20631 non-null  int64  
 14  s20       20631 non-null  float64
 15  s21       20631 non-null  float64
 16  RUL       20631 non-null  int64  
dtypes: float64(15), int64(2)
memory usage: 2.7 MB


In [11]:
df.describe()

,setting1,setting2,s2,s3,s4,s7,s8,s9,s11,s12,s13,s14,s15,s17,s20,s21,RUL
count,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000,20631.000000
mean,-0.000009,0.000002,642.680934,1590.523119,1408.933782,553.367711,2388.096652,9065.242941,47.541168,521.413470,2388.096152,8143.752722,8.442146,393.210654,38.816271,23.289705,107.807862
std,0.002187,0.000293,0.500053,6.131150,9.000605,0.885092,0.070985,22.082880,0.267087,0.737553,0.071919,19.076176,0.037505,1.548763,0.180746,0.108251,68.880990
min,-0.008700,-0.000600,641.210000,1571.040000,1382.250000,549.850000,2387.900000,9021.730000,46.850000,518.690000,2387.880000,8099.940000,8.324900,388.000000,38.140000,22.894200,0.000000
25%,-0.001500,-0.000200,642.325000,1586.260000,1402.360000,552.810000,2388.050000,9053.100000,47.350000,520.960000,2388.040000,8133.245000,8.414900,392.000000,38.700000,23.221800,51.000000
50%,0.000000,0.000000,642.640000,1590.100000,1408.040000,553.440000,2388.090000,9060.660000,47.510000,521.480000,2388.090000,8140.540000,8.438900,393.000000,38.830000,23.297900,103.000000
75%,0.001500,0.000300,643.000000,1594.380000,1414.555000,554.010000,2388.140000,9069.420000,47.700000,521.950000,2388.140000,8148.310000,8.465600,394.000000,38.950000,23.366800,155.000000
max,0.008700,0.000600,644.530000,1616.910000,1441.490000,556.060000,2388.560000,9244.590000,48.530000,523.380000,2388.560000,8293.720000,8.584800,400.000000,39.430000,23.618400,361.000000


In [12]:
# import seaborn as sns
# sns.histplot(df['RUL'], kde=True, bins=10)
# df['RUL'].value_counts()

In [18]:
x = df.drop('RUL', axis=1)
y = df['RUL']
x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.2, random_state=42)
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [43]:
L = Lasso(random_state=42,max_iter=10000, tol=0.0002)
R = Ridge()
LR = LinearRegression()
EN = ElasticNet()
RF = RandomForestRegressor()
DT = DecisionTreeRegressor()
GB = GradientBoostingRegressor()
    alpha=1.0,
    *,
    fit_intercept=True,
    copy_X=True,
    max_iter=None,
    tol=0.0001,
    solver='auto',
    positive=False,
    random_state=None,

In [56]:
np.logspace(-10,2,100, base=2)
# print(9.76562500e-04)
# loguniform(1e-4, 100).rvs(10)
print(0.001/0.0001)

10.0


In [48]:
lasso_param = {
    'alpha': np.logspace(-10,2,100, base=2),
    'selection': ['cyclic', 'random'],
    'positive': [True, False]
}

ridge_param = {
    alpha: np.logspace(-10,2,100, base=2)
}

linear_regression _param = {
    
}

elastic_net_param = {
    
}

random_forest_param = {
    
}

decision_tree_param = {
    
}

gradient_boosting_param = {
    
}

SyntaxError: expression expected after dictionary key and ':' (1307977384.py, line 2)